In [ ]:
import sys, pandas
print(sys.executable)
print(pandas.__version__)

D:\TUGAS\C Path\Portfolio\Data Science\netflix-gap-analysis\.venv\Scripts\python.exe
3.0.6


In [3]:
   import pandas as pd, json
   RAW = "../data/raw/netflix_titles.csv"
   df = pd.read_csv(RAW, dtype=str, keep_default_na=True)
   profile = {}
   df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [4]:
print(df.shape)
df.columns.tolist()

(8807, 12)


['show_id',
 'type',
 'title',
 'director',
 'cast',
 'country',
 'date_added',
 'release_year',
 'rating',
 'duration',
 'listed_in',
 'description']

In [6]:
profile["rows"] = len(df)
profile["cols"] = df.shape[1]
profile["dup_show_id"] = int(df["show_id"].duplicated().sum())
profile["dup_title_type_year"] = int(df.duplicated(["title","type","release_year"]).sum())
profile["dup_full_rows"] = int(df.duplicated().sum())
profile

{'rows': 8807,
 'cols': 12,
 'dup_show_id': 0,
 'dup_title_type_year': 0,
 'dup_full_rows': 0}

In [7]:
nulls = df.isna().sum()
profile["nulls"] = nulls.to_dict()
profile["null_pct"] = (nulls / len(df) * 100).round(1).to_dict()
pd.DataFrame({"nulls": nulls, "pct": profile["null_pct"]}).sort_values("nulls", ascending=False)

,nulls,pct
director,2634,29.9
country,831,9.4
cast,825,9.4
date_added,10,0.1
rating,4,0.0
duration,3,0.0
show_id,0,0.0
type,0,0.0
title,0,0.0
release_year,0,0.0


In [8]:
# 1. all distinct rating values
print(df["rating"].value_counts(dropna=False))

rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NaN            4
NC-17          3
UR             3
74 min         1
84 min         1
66 min         1
Name: count, dtype: int64


In [9]:
# 2. rows with a duration value sitting in the rating column
bad_rating = df[df["rating"].str.contains("min", na=False)]
profile["rating_has_minutes"] = len(bad_rating)
bad_rating[["show_id","title","type","rating","duration"]]

,show_id,title,type,rating,duration
5541,s5542,Louis C.K. 2017,Movie,74 min,NaN
5794,s5795,Louis C.K.: Hilarious,Movie,84 min,NaN
5813,s5814,Louis C.K.: Live at the Comedy Store,Movie,66 min,NaN


In [10]:
# 3. duration units
print(df["duration"].str.extract(r"([A-Za-z]+)$")[0].value_counts(dropna=False))

0
min        6128
Season     1793
Seasons     883
NaN           3
Name: count, dtype: int64


In [11]:
# Which 4 titles have a truly null rating?
df[df["rating"].isna()][["show_id","title","type","release_year","duration"]]

,show_id,title,type,release_year,duration
5989,s5990,13TH: A Conversation with Oprah Winfrey & Ava ...,Movie,2017,37 min
6827,s6828,Gargantia on the Verdurous Planet,TV Show,2013,1 Season
7312,s7313,Little Lunch,TV Show,2015,1 Season
7537,s7538,My Honor Was Loyalty,Movie,2015,115 min


In [12]:
# Cross-check: units should match type
df["duration_unit"] = df["duration"].str.extract(r"([A-Za-z]+)$")[0]
print(pd.crosstab(df["type"], df["duration_unit"], dropna=False))

duration_unit  Season  Seasons   min  NaN
type                                     
Movie               0        0  6128    3
TV Show          1793      883     0    0


In [14]:
# Save the numbers
profile["rating_nunique_raw"] = int(df["rating"].nunique())
profile["rating_null"] = int(df["rating"].isna().sum())
profile["duration_null"] = int(df["duration"].isna().sum())
profile["rating_NR_UR"] = int(df["rating"].isin(["NR","UR"]).sum())

In [15]:
d = df["date_added"].dropna()
profile["date_added_leading_trailing_space"] = int((d != d.str.strip()).sum())

parsed = pd.to_datetime(d.str.strip(), format="%B %d, %Y", errors="coerce")
profile["date_added_unparseable"] = int(parsed.isna().sum())
profile["date_added_year_min"] = int(parsed.dt.year.min())
profile["date_added_year_max"] = int(parsed.dt.year.max())

d.head(3).tolist()

['September 25, 2021', 'September 24, 2021', 'September 24, 2021']

In [17]:
# Added to Netflix before the title's release year? (odd but possible for licensed/older content, so flag only)
tmp = df.loc[parsed.index].copy()
tmp["added_year"] = parsed.dt.year
profile["added_before_release_year"] = int((tmp["added_year"] < tmp["release_year"].astype(int)).sum())

In [18]:
print({k: v for k, v in profile.items() if k.startswith(("date_added", "added_before"))})
print(d.head(3).map(repr).tolist())   # repr() makes leading spaces visible

{'date_added_leading_trailing_space': 88, 'date_added_unparseable': 0, 'date_added_year_min': 2008, 'date_added_year_max': 2021, 'added_before_release_year': 14}
["'September 25, 2021'", "'September 24, 2021'", "'September 24, 2021'"]


In [19]:
print(d[d != d.str.strip()].map(repr).head(5).tolist())

["' August 4, 2017'", "' December 23, 2018'", "' December 15, 2018'", "' July 1, 2017'", "' July 26, 2019'"]


In [20]:
tmp[tmp["added_year"] < tmp["release_year"].astype(int)][["title","release_year","date_added"]].head(14)

,title,release_year,date_added
1551,Hilda,2021,"December 14, 2020"
1696,Polly Pocket,2021,"November 15, 2020"
2920,Love Is Blind,2021,"February 13, 2020"
3168,Fuller House,2020,"December 6, 2019"
3287,Maradona in Mexico,2020,"November 13, 2019"
3369,BoJack Horseman,2020,"October 25, 2019"
3433,The Hook Up Plan,2020,"October 11, 2019"
4844,Unbreakable Kimmy Schmidt,2019,"May 30, 2018"
4845,Arrested Development,2019,"May 29, 2018"
5394,Hans Teeuwen: Real Rancour,2018,"July 1, 2017"


In [21]:
for col in ["country", "listed_in", "cast", "director"]:
    s = df[col].dropna()
    profile[f"{col}_multi_value_rows"] = int(s.str.contains(",").sum())
    profile[f"{col}_max_items"] = int(s.str.split(",").str.len().max())

# country + genre: split, strip, then look for real problems
for col in ["country", "listed_in"]:
    items = df[col].dropna().str.split(",").explode()
    stripped = items.str.strip()
    profile[f"{col}_empty_items"] = int((stripped == "").sum())      # trailing commas
    profile[f"{col}_distinct_raw"] = int(items.nunique())            # before strip
    profile[f"{col}_distinct_clean"] = int(stripped[stripped != ""].nunique())

# other text columns: any stray leading/trailing spaces?
for col in df.columns:
    s = df[col].dropna()
    n = int((s != s.str.strip()).sum())
    if n: print(col, n)

# Indonesia sample size, because it decides how you present that page
c_clean = df["country"].dropna().str.split(",").explode().str.strip()
profile["indonesia_titles"] = int((c_clean == "Indonesia").sum())

print({k: v for k, v in profile.items() if k.startswith(("country", "listed_in", "cast", "director", "indonesia"))})

title 1
date_added 88
{'country_multi_value_rows': 1320, 'country_max_items': 12, 'listed_in_multi_value_rows': 6787, 'listed_in_max_items': 3, 'cast_multi_value_rows': 7101, 'cast_max_items': 50, 'director_multi_value_rows': 614, 'director_max_items': 13, 'country_empty_items': 7, 'country_distinct_raw': 197, 'country_distinct_clean': 122, 'listed_in_empty_items': 0, 'listed_in_distinct_raw': 73, 'listed_in_distinct_clean': 42, 'indonesia_titles': 90}


In [22]:
df[df["rating"].isna()][["show_id","title","type","release_year","duration"]]

,show_id,title,type,release_year,duration
5989,s5990,13TH: A Conversation with Oprah Winfrey & Ava ...,Movie,2017,37 min
6827,s6828,Gargantia on the Verdurous Planet,TV Show,2013,1 Season
7312,s7313,Little Lunch,TV Show,2015,1 Season
7537,s7538,My Honor Was Loyalty,Movie,2015,115 min


In [23]:
df["duration_unit"] = df["duration"].str.extract(r"([A-Za-z]+)$")[0]
print(pd.crosstab(df["type"], df["duration_unit"], dropna=False))
df = df.drop(columns="duration_unit")

duration_unit  Season  Seasons   min  NaN
type                                     
Movie               0        0  6128    3
TV Show          1793      883     0    0


In [28]:
# 1. title with stray space (repr makes the space visible)
bad_title = df["title"] != df["title"].str.strip()
print("Titles with stray spaces:")
print(df.loc[bad_title, ["show_id"]].assign(title=df.loc[bad_title, "title"].map(repr)).to_string())

# 2. the 4 null-rating titles
print("\nNull-rating titles:")
print(df[df["rating"].isna()][["show_id","title","type","release_year","duration"]].to_string())

# 3. type x duration unit crosstab
unit = df["duration"].str.extract(r"([A-Za-z]+)$")[0]
print("\nType x duration unit:")
print(pd.crosstab(df["type"], unit, dropna=False))

# 4. top countries + all distinct names, to scan for variants
print("\nTop 15 countries:")
print(c_clean.value_counts().head(15).to_string())
print("\nAll distinct countries:")
print(sorted(c_clean.unique()))

# 5. save the profile (with-block closes the file properly)
os.makedirs("../docs", exist_ok=True)
with open("../docs/profile_before.json", "w") as f:
    json.dump(profile, f, indent=2)
print("\nSaved profile_before.json with", len(profile), "keys")

Titles with stray spaces:
     show_id               title
6529   s6530  'Consequences\xa0'

Null-rating titles:
     show_id                                                   title     type release_year  duration
5989   s5990  13TH: A Conversation with Oprah Winfrey & Ava DuVernay    Movie         2017    37 min
6827   s6828                       Gargantia on the Verdurous Planet  TV Show         2013  1 Season
7312   s7313                                            Little Lunch  TV Show         2015  1 Season
7537   s7538                                    My Honor Was Loyalty    Movie         2015   115 min

Type x duration unit:
0        Season  Seasons   min  NaN
type                               
Movie         0        0  6128    3
TV Show    1793      883     0    0

Top 15 countries:
country
United States     3690
India             1046
United Kingdom     806
Canada             445
France             393
Japan              318
Spain              232
South Korea        231
Germ

In [27]:
import os, json
os.makedirs("../docs", exist_ok=True)
with open("../docs/profile_before.json", "w") as f:
    json.dump(profile, f, indent=2)
print("Saved profile_before.json with", len(profile), "keys")

Saved profile_before.json with 32 keys
